# Object Builder Documentation

## Introduction to Object Builder
This report introduces a new CIM-Builder library named "Object Builder", that enables end user to add new CIM-based equipments to existing test system data in a reliable manner. An overview of the work flow of the Object Builder is shown in Figure 1.
<p>
<!--<img src="Report_Figures/Robot_in_Workshop.png" alt="My Plot" align="right" width="10%"/>-->
The library takes as input the original eXtensible Markup Language (XML) based model along with the specificaiton of the new element to be added. The object builder integrates the requested element into the appropriate location within the network, ensuring consistency with CIM-semantics and power system operational requirements. 
<img src="Report_Figures/Object_Builder_Overview_Figure.png" alt="My Plot" align="left" style="margin: 20px; width: 40%"/>
The resulting network is then output as a modified XML file. This process allows end users to construct Grid Atlas models in a streamlined, efficient, and CIM-compliant manner.
<!--</p>
<br><br>
<p>-->
</p>


Each individual components within the object builder is explained using individual subsections. Later, how each of them connects with each other to form the complete working object builder will be explained.

## Catalog Parser

Catalog Parser is the core of the object builder. Catalog Parser takes in the pre-processed and structured data from Easy-CIM anonymized using Diff Privacy. Input file in json format is converted to catalog dictionaries first. The dictionary is passsed to item parser which extracts attributes and values for each of the components in the input network.

An illustration of working of Catalog Parser is provided below using PowerTransformer object as an exmaple.

STEP 1: Initialization and loading network from XML file

In [1]:
import os
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
import cimgraph.data_profile.cimhub_2023 as cim
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
file = XMLFile(filename='../test_models/IEEE13.xml')
feeder = cim.Feeder(mRID= '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
container = feeder
network = FeederModel(container=feeder, connection=file)

STEP 2: Run Catalog Parser and get Object Output, this will serve as input to the New_New_Power_Transformer function

In [2]:
import logging
import json 
from cimgraph.models import GraphModel
from __future__ import annotations


from cimgraph.databases import get_cim_profile
_log = logging.getLogger(__name__)

def catalog_parser(catalog_file, network):
    file = open(catalog_file) ## opening json file
    catalog = json.load(file) ## loading jsonj file contents as catalog
    data = catalog['catalog'] ## catalog dictionary, first element (in this example, only single XF) saved into data
    cim_profile, cim = get_cim_profile() # Import CIM profile 
    obj = item_parser(data, network, cim) ## validating and simplifying json file objects into a single structure of obj. - needs modifications
    file.close()
    return obj

def item_parser(data:dict, network: GraphModel, cim):
    class_type = edge_class = eval(f'cim.{data["@type"]}')
    obj = class_type()
    network.add_to_graph(obj)

    for attribute in data:
        if type(data[attribute]) == str:
            if attribute in class_type.__dataclass_fields__:
                setattr(obj, attribute, data[attribute])
            else:
                _log.warning(f'Attribute {attribute} not found')
        elif type(data[attribute]) == list:
            if attribute in class_type.__dataclass_fields__:
                values = getattr(obj, attribute)
                for item in data[attribute]:
                    value = item_parser(item, network, cim)
                    values.append(value)
                setattr(obj, attribute, values)
    return obj

Catalog_JSON_file_path = '../test_models/hv69_12.json' ### Power Transformer json file path
Obj_CP_output = catalog_parser(Catalog_JSON_file_path, network) ### Catalog file reads the input and returns an object 

Attribute @type not found
Attribute @type not found
Attribute x not found
Attribute b not found
Attribute @type not found
Attribute x not found
Attribute b not found


Step 3: Save the updated network to xml file. 

In [3]:
import os
print(os.getcwd())
from cimgraph import utils
utils.write_xml(network=network, filename='IEEE_13_with_Xfmr_CPOB_0702.xml')

/root/CIM-Builder/tests/test_scripts


## Verification of Object Builder Performance

Object builder adds the specified new equipments at the specified locations in the given netowrk. The output is the updated netowrk in the xml format. As XML files are not particulary friendly for human eyes, an evaluation script that could summarize the changes made to the XML files by Object Builder in an easy to understand form is developed. 

The evaluation script is named CIMparator as it comapres two CIM-based xmls and reports back the differences - additions, removals as well as change in field values. The main component of the CIMparator is the CIMparser script that parses an input XML file as an XML tree structure and extracts the classes, attributes and the field values in a structured fashion. A CIMparser is applied to both inputs - original netowrk and modified network - and the resulting outputs are compared to produce eventual outputs for CIMparator. An example usage of the CIMparator is provided next.

In [7]:
from lxml import etree
from collections import defaultdict
import os
import csv

In [8]:

# === Normalize values for robust comparison ===
def normalize_value(val):
    if val is None:
        return ""
    val = val.strip()
    if val.lower() == "true":
        return True
    if val.lower() == "false":
        return False
    try:
        return float(val)
    except ValueError:
        return val.lower()  # case-insensitive string comparison

# === Parse CIM XML ===
def parse_cim(xml_file):
    ns = {
        'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
        'cim': 'http://iec.ch/TC57/CIM100#'
    }
    tree = etree.parse(xml_file)
    root = tree.getroot()

    class_data = defaultdict(dict)  # {class: {instance_id: {attr: value}}}
    for element in root:
        if not isinstance(element.tag, str):
            continue
        cls = etree.QName(element.tag).localname
        inst_id = element.get('{%s}ID' % ns['rdf']) or element.get('{%s}about' % ns['rdf'])

        attr_map = {}
        for child in element:
            if not isinstance(child.tag, str):
                continue
            attr = etree.QName(child.tag).localname
            attr_value = child.text or child.get('{%s}resource' % ns['rdf'])
            attr_map[attr] = attr_value

        class_data[cls][inst_id] = attr_map

    return class_data

# === Compare CIM Files ===
def compare_cim_files(data1, data2):
    all_classes = set(data1.keys()) | set(data2.keys())
    added, removed, modified = [], [], []

    for cls in sorted(all_classes):
        ids1 = data1.get(cls, {})
        ids2 = data2.get(cls, {})
        all_ids = set(ids1.keys()) | set(ids2.keys())

        for inst_id in sorted(all_ids):
            if inst_id not in ids1:
                added.append((cls, inst_id, ids2[inst_id]))
            elif inst_id not in ids2:
                removed.append((cls, inst_id, ids1[inst_id]))
            else:
                attrs1 = ids1[inst_id]
                attrs2 = ids2[inst_id]
                changes = {}
                all_attrs = set(attrs1.keys()) | set(attrs2.keys())
                for attr in all_attrs:
                    v1 = normalize_value(attrs1.get(attr))
                    v2 = normalize_value(attrs2.get(attr))
                    if v1 != v2:
                        changes[attr] = (attrs1.get(attr), attrs2.get(attr))
                if changes:
                    modified.append((cls, inst_id, changes))
    return added, removed, modified

# === Save to CSV ===
def save_added_removed(file_path, data, change_type):
    with open(file_path, "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Class", "Instance ID", "Attribute", "Value", "Change Type"])
        for cls, inst_id, attrs in data:
            for attr, val in attrs.items():
                writer.writerow([cls, inst_id, attr, val, change_type])

def save_modified(file_path, data):
    with open(file_path, "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Class", "Instance ID", "Attribute", "Old Value", "New Value"])
        for cls, inst_id, changes in data:
            for attr, (old, new) in changes.items():
                writer.writerow([cls, inst_id, attr, old, new])


In [10]:

# === MAIN ===
if __name__ == "__main__":
    file1 = "../test_models/IEEE13.xml"
    file2 = "../test_scripts/IEEE_13_with_Xfmr_CPOB.xml"

    print("Parsing files...")
    data1 = parse_cim(file1) 
    data2 = parse_cim(file2)

    print("Comparing files...")
    added, removed, modified = compare_cim_files(data1, data2)

    # Save CSVs
    save_added_removed("cim_differences_added.csv", added, "ADDED")
    save_added_removed("cim_differences_removed.csv", removed, "REMOVED")
    save_modified("cim_differences_modified.csv", modified)

    # === Final Summary ===
    print("\n=== Summary ===")
    print(f"Added instances: {len(added)}")
    print(f"Removed instances: {len(removed)}")
    print(f"Modified instances: {len(modified)}")

    print("\nCSV files saved:")
    print(" - cim_differences_added.csv")
    print(" - cim_differences_removed.csv")
    print(" - cim_differences_modified.csv")


Parsing files...
Comparing files...

=== Summary ===
Added instances: 3
Removed instances: 0
Modified instances: 35

CSV files saved:
 - cim_differences_added.csv
 - cim_differences_removed.csv
 - cim_differences_modified.csv
